![imagenes](logo.png)

# Espacios normados y con producto interno
## Midiendo intensidad y parecido entre acordes

En el capítulo anterior construimos el universo infinito de combinaciones posibles de teclas: el espacio vectorial. Puedes sumar acordes, escalarlos (tocarlos más fuerte o más suave), pero aún nos falta responder preguntas esenciales:

- ¿Qué tan **fuerte** suena ese acorde en total? (intensidad, volumen, energía)
- ¿Qué tan **parecidos** son dos acordes? ¿Se refuerzan o se cancelan?
- ¿Cuánto "ángulo" hay entre dos melodías representadas como vectores?

Para eso dotamos al espacio de herramientas de **medición**: la **norma** (para medir tamaño/longitud) y el **producto interno** (para medir ángulos, similitud y proyecciones).

Y la joya que une todo: la **desigualdad de Cauchy-Schwarz**, que básicamente dice:  
**"el parecido entre dos cosas nunca puede superar el producto de sus intensidades individuales"**.

Es la base matemática de la similitud coseno, la estabilidad en regresión, la atención en transformers y casi todo lo que usamos en machine learning cuando hablamos de vectores.

In [ ]:
# Importamos las librerías necesarias
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D
from sklearn.metrics.pairwise import cosine_distances, cosine_similarity

# Configuración para gráficos más bonitos
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("✅ Librerías cargadas correctamente")

## La norma: ¿qué tan fuerte suena ese acorde?

La **norma** mide la "intensidad total" o "energía" de un vector. En nuestra analogía musical: si este acorde es una mezcla de notas, ¿cuánto volumen genera en conjunto?

### Definición formal

Una **norma** $\|\cdot\|$ es una función que asigna a cada vector un número ≥ 0 y cumple tres reglas:

| Propiedad | Fórmula | Analogía musical |
|-----------|---------|------------------|
| No negatividad | $\lVert v \rVert \ge 0$ y $\lVert v \rVert = 0 \iff v = 0$ | No hay acorde que suene sin tener notas |
| Homogeneidad | $\lVert c v \rVert = |c| \lVert v \rVert$ | Duplicar la intensidad duplica el volumen |
| Desigualdad triangular | $\lVert u + v \rVert \le \lVert u \rVert + \lVert v \rVert$ | Dos acordes juntos no suenan más fuerte que por separado |

En la Desigualdad del triángulo se cumple que $\|u+v\|=\|u\|+\|v\|$ si y solo si $u=av$ para algún escalar $a$


### La norma más usada: Euclidiana (L²)

$$
\| \mathbf{v} \|_2 = \sqrt{v_1^2 + v_2^2 + \dots + v_n^2}
$$

In [ ]:
vector1 = np.array([1.0, 0.8, 1.2]) 
vector2 = np.array([0.9, 1.1, 0.0])   

norma_vec1 = np.linalg.norm(vector1)
norma_vec2 = np.linalg.norm(vector2)

In [ ]:
norma_vec1

In [ ]:
# Demostración de propiedades

#Homogeneidad: 
np.linalg.norm(2*vector1)

In [ ]:
# Desigualdad Triangular:
np.linalg.norm(vector1 + vector2) <= norma_vec1 + norma_vec2

**Preguntas**

1. Si triplicas todas las coordenadas de un vector, ¿qué pasa con su norma?
2. Si cambias el signo de todas las coordenadas de un vector, ¿cómo cambia la norma?
3. ¿Puede la norma de un vector ser negativa? ¿Por qué?

## Producto interno: midiendo cuánto se "alinean" dos acordes

El **producto interno** (producto punto) captura similitud direccional y ángulo entre vectores.

En $\mathbb{R}^n$: tomemos $u=(u_1,u_2,...,u_n)$ y $v=(v_1,v_2,...,v_n)$

$$
\mathbf{u} \cdot \mathbf{v} = u_1 v_1 + u_2 v_2 + \cdots + u_n v_n
$$

Y geométricamente:

$$
\mathbf{u} \cdot \mathbf{v} = \|\mathbf{u}\| \, \|\mathbf{v}\| \cos \theta
$$

La **similitud coseno** normaliza para que el resultado esté entre $-1$ y $1$:

$$
\cos \theta = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \, \|\mathbf{v}\|}
$$

Esto es el corazón de **embeddings, búsqueda semántica, recomendaciones**, etc.

In [ ]:
# Ejemplo con embeddings ficticios de palabras
gato = np.array([1.0, 2.0, 3.0, 1.0])      # embedding de "gato"
felino = np.array([2.0, 4.0, 6.0, 2.0])    # embedding de "felino" (muy similar)
perro = np.array([1.0, 1.5, 2.5, 3.0])     # embedding de "perro" (algo similar)
coche = np.array([5.0, 0.0, 1.0, 8.0])     # embedding de "coche" (diferente)


prod_gato_felino = np.dot(gato,felino)
norma_gato = np.linalg.norm(gato)
norma_felino = np.linalg.norm(felino)

similitud_gato_felino = prod_gato_felino / (norma_gato*norma_felino )
similitud_gato_felino

### ⚠️ Confusión común

La similitud coseno mide **dirección**, no **magnitud**. Dos vectores [1,2,3] y [10,20,30] tienen coseno = 1 (perfectamente alineados) aunque sus normas sean muy diferentes.

En música: dos canciones con la misma "forma" pero una mucho más fuerte suenan "iguales" en términos de patrón, aunque una sea más ruidosa.

In [ ]:
# Demostración: vectores con misma dirección pero diferente magnitud
v1 = np.array([1, 2, 3])
v2 = 5 * v1  # [5, 10, 15]

dot = np.dot(v1, v2)
cos_sim = dot / (np.linalg.norm(v1) * np.linalg.norm(v2))

print(f"v1 = {v1}, ‖v1‖ = {np.linalg.norm(v1):.2f}")
print(f"v2 = {v2}, ‖v2‖ = {np.linalg.norm(v2):.2f}")
print(f"Similitud coseno: {cos_sim:.4f} (¡perfectamente alineados!)")

## El producto interno como ente abstracto

Dado un espacio vectorial sobre **los reales**, se dice que una operación, denotada por $\langle\cdot,\cdot\rangle$, que toma dos vectores y devuelve un número real es un producto interno si 

- $\langle u,v\rangle=\langle v,u\rangle$ para cualesquiera dos vectores $u$ y $v$
- $\langle au,v\rangle = a\langle u,v\rangle$ para cualquier escalar $a$
- $\langle u+w,v\rangle = \langle u,v\rangle+\langle w,v\rangle$ para cualesquiera vectores $u$, $v$, $w$
- $\langle u,u\rangle\ge 0$ y $\langle u,u\rangle=0$ si y solo si $u=\boldsymbol{0}$

**IMPORTANTE.** Si $\langle\cdot,\cdot\rangle$ es un producto en el espacio vectorial $V$, entonces la fórmula $\sqrt{\langle u,u\rangle}$ define una norma en $V$. Es decir, $\|u\|=\sqrt{\langle u,v\rangle}$

## La desigualdad de Cauchy–Schwarz: el límite natural del parecido

**Desigualdad de Cauchy–Schwarz**

En cualquier espacio con producto interno:

$$
|\langle \mathbf{u},\mathbf{v}\rangle| \le \|\mathbf{u}\|\,\|\mathbf{v}\|
$$

Con igualdad si y solo si $\mathbf{u}$ y $\mathbf{v}$ son paralelos (uno es múltiplo escalar del otro, o alguno es cero).

En palabras simples: **el parecido crudo nunca supera el producto de las intensidades**. No puedes tener más armonía de la que permiten las fuerzas individuales.

### Intuición geométrica

Del producto punto sabemos que:

$$
\mathbf{u}\cdot\mathbf{v} = \|\mathbf{u}\|\,\|\mathbf{v}\|\cos\theta
\quad \Rightarrow \quad
|\mathbf{u}\cdot\mathbf{v}| =
\|\mathbf{u}\|\,\|\mathbf{v}\|\,|\cos\theta|
\le
\|\mathbf{u}\|\,\|\mathbf{v}\|
$$

Porque $|\cos\theta|\le 1$ siempre.

### ✏️ Autoevaluación

¿Puede la similitud coseno entre dos vectores ser 1.5? ¿Por qué?

## Ortogonalidad: acordes que no se interfieren

Dos vectores son **ortogonales** si:

$$
\mathbf{u}\cdot\mathbf{v} = 0
$$

(ángulo de $90^\circ$)

En datos: **variables ortogonales aportan información completamente independiente**, lo cual es ideal para modelos estables.

Ejemplo clásico: las **bases canónicas** $e_1, e_2, \dots, e_n$ son mutuamente ortogonales.

In [ ]:
# Bases canónicas en 3D
e1 = np.array([1, 0, 0])
e2 = np.array([0, 1, 0])
e3 = np.array([0, 0, 1])

### ⚠️ Confusión común

La ortogonalidad (producto interno = 0) no significa "falta de relación" en el sentido coloquial. Significa que sus direcciones son perpendiculares. En datos: dos variables ortogonales son linealmente independientes, pero pueden tener relaciones no lineales.

## 5. Proyecciones: aislar la parte "relevante" de un acorde

La **proyección de** $\mathbf{u}$ **sobre** $\mathbf{v}$ es la componente de $\mathbf{u}$ en la dirección de $\mathbf{v}$:

$$
\operatorname{proj}_{\mathbf{v}} \mathbf{u}
=
\left(
\frac{\mathbf{u}\cdot\mathbf{v}}{\|\mathbf{v}\|^2}
\right)
\mathbf{v}
$$

En **ML**: se usa en **regresión (least squares)**, **PCA**, **residuales**, etc.

In [ ]:
# Ejemplo
u = np.array([3.0, 4.0, 5.0])
v = np.array([1.0, 0.0, 1.0])   # dirección de interés

proj = (np.dot(u,v)/np.linalg.norm(v)**2)*v
perpendicular = u - proj  # componente ortogonal

In [ ]:
proj

## 6. Espacios métricos: el siguiente nivel

Hasta ahora hemos medido el **tamaño** de un vector (norma) y su **parecido/ángulo** (producto interno). Pero a veces lo que realmente nos interesa es: **¿qué tan lejos está un punto de otro?**

Un **espacio métrico** nos da una forma consistente de calcular distancias entre cualquier par de puntos.

### Definición formal

Un espacio métrico es un conjunto $V$ con una función **distancia** $d: V \times V \to \mathbb{R}$ que cumple:

| Propiedad | Fórmula | Analogía |
|-----------|---------|----------|
| No negatividad | $d(x,y) \geq 0$ | Las distancias no son negativas |
| Identidad | $d(x,y)=0 \iff x=y$ | Solo estás a distancia cero de ti mismo |
| Simetría | $d(x,y) = d(y,x)$ | La distancia de A a B es igual que de B a A |
| Desigualdad triangular | $d(x,z) \leq d(x,y) + d(y,z)$ | Ir directo es siempre el camino más corto |

## 7. Distancias más usadas en Machine Learning

### 7.1 Distancia Euclidiana (L²)

La clásica: "distancia en línea recta". Inducida por la norma L².

$$
d_{euclid}(\mathbf{x}, \mathbf{y}) = \|\mathbf{x} - \mathbf{y}\|_2 = \sqrt{\sum_{i=1}^n (x_i - y_i)^2}
$$

### 7.2 Distancia Manhattan (L¹, "distancia del taxista")

Mide la distancia recorriendo ejes por separado. Útil cuando **los cambios en cada dimensión importan por separado** (ej. presupuestos, conteos, datos dispersos).

$$
d_{manh}(\mathbf{x},\mathbf{y}) = \sum_{i=1}^{n} |x_i - y_i|
$$

### 7.3 Distancia Coseno

Mide el ángulo entre vectores, ignorando su magnitud. Se define como:

$$
d_{cos}(\mathbf{x}, \mathbf{y}) = 1 - \cos(\theta) = 1 - \frac{\mathbf{x} \cdot \mathbf{y}}{\|\mathbf{x}\| \|\mathbf{y}\|}
$$

Va de 0 (vectores idénticos en dirección) a 2 (vectores opuestos).

## 8. ¿Por qué son importantes las métricas en Machine Learning?

Las métricas están en el corazón de casi todos los algoritmos de ML:

| Algoritmo | Métrica típica | ¿Para qué la usa? |
|-----------|----------------|-------------------|
| **K-Nearest Neighbors (KNN)** | Euclidiana, Manhattan | Encontrar los k puntos más cercanos |
| **K-Means** | Euclidiana | Asignar puntos al centroide más cercano |
| **DBSCAN** | Euclidiana | Identificar regiones densas |
| **Recomendadores** | Coseno | Encontrar ítems o usuarios similares |
| **PCA** | Euclidiana (implícita) | Maximizar varianza (distancias al cuadrado) |
| **Regresión** | Euclidiana al cuadrado | Minimizar error cuadrático |
| **Árboles de decisión** | No necesitan métrica | Divisiones por características |